https://github.com/D4Vinci/Scrapling/tree/main

In [1]:
import collections
import logging
import pandas as pd
import re
from scrapling.fetchers import AsyncDynamicSession, AsyncStealthySession
from playwright.async_api import Page

logging.getLogger("scrapling").setLevel(logging.WARNING)

Quick test on real website - Zara, which is highly dynamic

In [3]:
async def get_zara_page(page = "https://www.zara.com/es/"):
    async with AsyncDynamicSession(headless = True) as session:
        page = await session.fetch(page)
        return page

zara_page = await get_zara_page()

In [4]:
def is_in_category(category_name, link):
    translations = {
        "hombre": ["man"],
        "mujer": ["woman"],
        "ninos": ["kids"],
        # "accesorios": ["accessories"],
        # "calzado": ["shoes"],
        # "colecciones": ["collections"],
    }
    related_categories = translations.get(category_name, []) + [category_name]
    for related_category in related_categories:
        if re.search(related_category, link, re.IGNORECASE):
            return related_category


def categorise_all_links(main_page: Page):
    categories_dict = collections.defaultdict(list)
    categories = main_page.css("a.layout-categories-category--level-1")
    category_values = main_page.css(
        "div.zds-carousel.zds-tabs__panel-list.layout-categories__tabs-panel-list-wrapper"
    )
    category_values = category_values.css("a")

    found_category_links = set()
    for category in categories:
        category_name = category.css(".layout-categories-category-name")[0].text.lower().replace("ñ", "n")
        for cat in category_values:
            if not cat.attrib.get("href"):
                continue
            category_link = cat.attrib.get("href")
            if is_in_category(category_name, category_link) and category_link not in found_category_links:
                categories_dict[category_name].append(category_link)
                found_category_links.add(category_link)

    remaining_category_links = set(cat.attrib.get("href") for cat in category_values) - found_category_links
    return categories_dict, remaining_category_links


categories_dict, remaining_category_links = categorise_all_links(zara_page)
# print("Example category and links:")
# for category, links in categories_dict.items():
#     print(f"{category}: {links[:3]}")  # print only first 3 links for brevity

In [5]:
PRODUCT_SELECTOR = "li.product-grid-product._product"

async def scroll_to_load_all_products(page: Page):
    prev = 0
    while True:
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(2500)
        count = await page.evaluate(
            f"document.querySelectorAll('{PRODUCT_SELECTOR}').length"
        )
        if count == prev:
            break
        prev = count

async def get_zara_page_all_products(url: str):
    async with AsyncDynamicSession(headless = True) as session:
        page = await session.fetch(url, page_action = scroll_to_load_all_products)
        return page

woman_link_all = await get_zara_page_all_products(categories_dict["mujer"][0])
products = woman_link_all.css(PRODUCT_SELECTOR)
print(f"Total products: {len(products)}")

Total products: 854


In [6]:
products = []
for product in woman_link_all.css(PRODUCT_SELECTOR):
    product_id = product.attrib.get("data-productid")
    product_link = product.css("a.product-link")
    product_link = product_link[0].attrib.get("href") if product_link else None
    products.append((product_id, product_link))
products = pd.DataFrame(products, columns=["product_id", "product_link"])
products = products.dropna(subset = ["product_id"])

In [7]:
example_product_page = await get_zara_page(products.iloc[0]["product_link"])
product_infos = example_product_page.css("div[class^='product-detail-view'][class*='info']")
print("\n=== Product Information ===")
for i, info in enumerate(product_infos):
    if info.css("div[class*='main']"):
        header_name = info.css("h1[class*='header-name']")
        header_text = header_name[0].text.strip() if header_name else f"info_{i}"
        price = info.find_by_regex(r"\d+[\.,]?\d*\s*EUR", first_match = True)
        print(f"Header: {header_text}, Price: {price.text if price else 'N/A'}")
    elif info.css("div[class*='secondary']"):
        composition = info.css("li[class*='composition']")
        for i, comp in enumerate(composition):
            part_name = comp.css("span[class*='part-name']")[0].text.strip()\
                if comp.css("span[class*='part-name']") else None
            pcts = comp.css("ul").css("li")
            pcts = [p.text.strip() for p in pcts] if pcts else []
            composition_text = comp.text.strip() if comp else f"additional_info_{i}"
            print(f"Part: {part_name if part_name else 'N/A'}, Percentages: {pcts if pcts else 'N/A'}")


=== Product Information ===
Header: BLAZER ENTALLADA ASIMÉTRICA HOMBRERAS, Price: 59,95 EUR
Part: EXTERIOR, Percentages: ['74% poliéster', '19% viscosa', '7% elastano']
Part: FORRO, Percentages: ['100% acetato']
